In [ ]:
import scipy
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

# Para clustering
from sklearn.datasets import make_blobs
from sklearn.datasets import make_circles
from sklearn.preprocessing import MinMaxScaler, StandardScaler 
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN

**Ejercicio 1**

Trabajamos con las variables "bill_length_mm" y "bill_depth_mm" de pinguinos.

1. Escalar las variables por Standard Scaler (KNN es sensible a las escalas)
2. Para cada uno de los pinguinos 15, 313 y 151 (ver slides), numerar en un gráfico los 9 pinguinos mas cercanos, ordenados de más cerca a más lejos.
3. Clasificar cada pingüino utilizando KNN con K = 1, 3 y 9.

In [ ]:
# Utilizamos NearestNeighbors para obtener los vecinos más cercanos
from sklearn.neighbors import NearestNeighbors

In [ ]:
# Eliminamos datos faltantes y reseteamos los índices, para no tener problemas al graficar
penguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)

In [ ]:
# Normalizamos las variables "bill_length_mm" y "bill_depth_mm" por MinMax
penguins[["bill_length_mm", "bill_depth_mm"]] = StandardScaler().???(penguins[["bill_length_mm", "bill_depth_mm"]])

penguins.head()

In [ ]:
# Nos quedamos solo con largo y profundidad del pico
datos = penguins[["bill_depth_mm", "bill_length_mm"]]
datos

In [ ]:
# Veamos primero los 9 vecinos más cercanos del pingüino 151
K = 9
ind = 15
neighbors = NearestNeighbors(n_neighbors=K+1)  # Esta función nos devuelve los más cercanos incluyendo a si mismo, por eso tomamos 10.
neighbors.fit(???)  # En el ajuste solamente almacenamos los datos

In [ ]:
# Cómo se guardan los datos??

# Podemos ver todos los atributos (variables) de una clase usando __dict__
neighbors.__dict__

In [ ]:
# Ahora podemos buscar los vecinos más cercanos a un punto cualquiera o un conjunto de puntos.
# Tenemos que pasarle un DataFrame
distances, indices = neighbors.kneighbors(datos.???)

In [ ]:
# Nos devuelve un vector de distancias (opcional)
distances

In [ ]:
# Lo convertimos a vector
distances.flatten()

Vemos que las distancias están ordenadas de menor a mayor.

In [ ]:
# Y un vector de índices

In [ ]:
indices.flatten()

In [ ]:
# Graficamos
# Utilizamos un nuevo canal para codificar información: texto o etiquetas en cada punto
(
    so.Plot(data = penguins.iloc[indices.flatten()], x = "bill_length_mm", y = "bill_depth_mm",  text = np.arange(K+1).astype(str))
    .add(so.Text(valign = "bottom"))
    .add(so.Dot(), color = ???)
    .add(so.Dot(color = "red"), data = penguins.iloc[[ind]], x = "bill_length_mm", y = "bill_depth_mm")
)

**Ejercicio 2**

1. Implementar una función que reciba un DataFrame (que tenga solo las variables numéricas a utilizar para medir distancias), un vector de categorías (indexado igual que el DataFrame), un índice y un valor de K y devuelva la predicción por KNN para el dato indicado. Importante: debemos ignorar al propio dato en la votación.
2. Aplicar la función a los datos de pingüinos.

**Sugerencia:** para elegir la categoría más votada podemos calcular la moda. El paquete statistics provee el comando `mode`.


In [ ]:
# Cargamos el comando mode
import statistics
from statistics import mode

In [ ]:
# Item 1

# Antes de hacer la función lo calculamos para un ejemplo.
K = 9
ind = 151
datos = penguins[["bill_depth_mm", "bill_length_mm"]]
categorias = penguins["species"]

In [ ]:
# Eliminamos al propio pinguino, para no incluirlo en los vecinos.
datos2 = datos.drop(index=ind)
categorias2 = categorias.drop(index=ind)

# Cargamos los datos en neighbors
# Tomamos K vecinos, porque ya eliminamos al propio pinguino
neighbors = NearestNeighbors(n_neighbors=K)
neighbors.fit(datos2)

# Los indices que devuelve kneighbors son posicion dentro del dataset (no indices del DF), 
# por eso tenemos que eliminar al pinguino[ind] de las categorias

In [ ]:
distances, indices = neighbors.kneighbors(datos.iloc[[ind]])
votos = ???       # Tener en cuenta que kneighbors devuelve posicion no indice
                  # Si es necesario, recordar usar flatten para convertir DF a Series

votos

In [ ]:
mode(votos)

In [ ]:
# Juntamos todo en una función
def mas_votado(datos, categorias, ind, K):

    # Eliminamos al propio pinguino, para no incluirlo en los vecinos.
    datos2 = datos.drop(index=ind)
    categorias2 = categorias.drop(index=ind)

    # Tomamos K vecinos, porque ya eliminamos al propio pinguino
    neighbors = NearestNeighbors(n_neighbors= K) 
    neighbors.fit(datos.drop([ind]))

    distances, indices = neighbors.kneighbors(datos.iloc[[ind]])
    votos = ???

    # Los indices que devuelve kneighbors son posicion dentro del dataset (no indices del DF), 
    # por eso tenemos que eliminar al pinguino[ind] de las categorias
    
    return(???)

In [ ]:
# Item 2 - Aplicamos la función a los datos de pingüinos
datos = penguins[["bill_depth_mm", "bill_length_mm"]]
categorias = penguins["species"]
mas_votado(datos, categorias, 313, 9)

**Ejercicio 3**
1. Implementar una función que reciba un DataFrame, un vector de categorías y un valor de K y calcule las predicciones para todos los datos y nos devuelva el porcentaje de aciertos. (Notar que estamos haciendo implicitamente validación leave-one-out.)
2. Aplicar la función a los datos de pingüinos.

In [ ]:
# Item 1
def knn_leave_one_out(datos, categorias, K):
    correctos = 0
    total = len(datos)
    for ind in range(total):
        prediccion = ???
        if(???):
            correctos += 1
    return(???)

In [ ]:
# Item 2 - Aplicamos la función a los datos de pingüinos
datos = penguins[["bill_depth_mm", "bill_length_mm"]]
categorias = penguins["species"]
knn_leave_one_out(datos, categorias, 9)

**Ejercicio 4**
Utilizando las funciones de los ejercicios anteriores, calcular el valor de $K$ (impar) óptimo para predecir la especie de un pingüino.

In [ ]:
for K in range(1,30,2):
    aciertos = knn_leave_one_out(datos, categorias, K)
    print(K, aciertos)

# Ejercicio: clasificación de dígitos escritos a mano (OCR)

El dataset Digits de scikit-learn es una versión reducida de un conjunto recolectado por el investigador E. Alpaydin y colaboradores en los años 90.

Las características principales son:

1. Son dígitos manuscritos (0–9).
2. Fueron escritos por distintas personas en formularios.
3. Las imágenes originales se procesaron y redujeron a una grilla de 8×8 píxeles.
4. Cada píxel toma valores entre 0 y 16, representando niveles de gris.
5. Hay 1797 imágenes en total.

Con el siguiente código podemos ver algunos ejemplos

In [ ]:
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

digits = load_digits()

fig, axes = plt.subplots(4, 10, figsize=(4,2))

for ax, img, label in zip(axes.ravel(),
                          digits.images[:40],
                          digits.target[:40]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

Usar la funcion `knn_leave_one_out` que implementamos para clasificar los dígitos buscando los números "más cercanos".

In [ ]:
# digits es un diccionario con mucha variables
digits

In [ ]:
# En data tenemos los datos como vectores, para podes hacer clasificación.
# En images tenemos los datos como matrices 8x8 para graficar.
digits.data.shape

In [ ]:
# En target tenemos la respuesta correcta
digits.target

In [ ]:
# Predecimos!
# Hace falta escalar en este caso?




# KNeighborsClassifier

Utilizamos ahora la función de sklean: KNeighborsClassifier

## Ejemplo: detección temprana de diabetes

A partir de distintos datos de pacientes queremos detectar tempranamente si ese paciente va a sufrir diabetes.

1. Leer los datos del archivo "diabetes.csv".
2. Separar la columna "Outcome" como variable respuesta y el resto como variables explicativas.
3. Partir la muestra en un 80% para entrenamiento y un 20% para testeo.
4. Escalar las variables explicativas por StandardScaler
5. Se quiere predecir la variable respuesta por KNN. Calcular el valor óptimo de $K$ optimizando el porcentaje de aciertos en testeo.
7. Para el valor hallado, calcular la matriz de confusión en testeo (es la matriz $C$ que guarda en la coordenada $C_{ij}$ la cantidad de observaciones en el grupo $i$ que fueron clasificadas en el grupo $j$).

In [ ]:
# Utilizamos estos paquetes
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
datos = pd.read_csv("diabetes.csv")
datos.head()

In [ ]:
# 2. Separamos la variable respuesta
X = datos.drop("Outcome",axis=1)
y = datos["Outcome"]

In [ ]:
# 3. Separamos en entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [ ]:
# 4. Escalamos X
scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.???
X_test_scaled = scaler.???

In [ ]:
neighbor = KNeighborsClassifier(n_neighbors=5)

In [ ]:
# Entrenamos utilizando X_train
neighbor.fit(???)

In [ ]:
# Predecimos utilizando X_test
y_pred = neighbor.predict(???)
y_pred

In [ ]:
# Calculamos la precisión con accuracy_score (porcentaje de aciertos)
print(accuracy_score(y_test,y_pred))

In [ ]:
# Como lo podemos hacer a mano?
??? / len(y_test)

In [ ]:
# Repetimos todo para varios valores de K
for K in range(1,30,2):
    neighbor = KNeighborsClassifier(n_neighbors=K)
    neighbor.fit(X_train_scaled,y_train)
    y_pred = neighbor.predict(X_test_scaled)
   
    print(K, accuracy_score(y_test,y_pred))

**Matriz de confusión**
La matriz de confusión $C$ guarda en la coordenada $C_{ij}$ la cantidad de observaciones en el grupo $i$ que fueron clasificadas en el grupo $j$.

Si la variable es binaria:
- $C_{00}$ son los casos negativos clasificados correctamente.
- $C_{01}$ son los casos negativos  clasificados como positivos (falsos positivos).
- $C_{10}$ son los casos positivos  clasificados como negativos (falsos negativos).
- $C_{11}$ son los casos positivos clasificados correctamente.

In [ ]:
K = 5
neighbor = KNeighborsClassifier(n_neighbors=K)
neighbor.fit(X_train_scaled,y_train)
y_pred = neighbor.predict(X_test_scaled)

In [ ]:
C = confusion_matrix(y_test,y_pred)
display(C)

**Ejercicio:** Calcular el coeficiente $C_{01}$ a mano.

In [ ]:
sum(???)

In [ ]:
# Si solo nos interesa evaluar el desempeño podemos usar score en lugar de predict
neighbor.fit(X_train_scaled,y_train).score(X_test_scaled,y_test)

In [ ]:
neighbor

**Ejercicio:** Utilizar KNeighborsClassifier con validación por leave-one-out.
En este caso podemos utilizar la función predict con parámetro None, y realiza las predicciones sobre el conjunto de entrenamiento, dejando siempre a la propia observacion afuera al hacer la predicción. Esto corresponde a validación leave-one-out.

In [ ]:
scaler = StandardScaler().set_output(transform="pandas")
X_scaled = scaler.fit_transform(X)

for K in range(1,30,2):
    neighbor = KNeighborsClassifier(n_neighbors=K)
    neighbor.fit(X_scaled,y)
    y_pred = neighbor.predict(None)
   
    print(K, ???)

**Ejemplo:** Calculamos la matriz de confusión para la clasificación de dígitos.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


# KNN
knn = KNeighborsClassifier(n_neighbors=5)
X=digits.data
y=digits.target
knn.fit(X,y)

# Predicción
y_pred = knn.predict(None)

acc = accuracy_score(y, y_pred)
print(f"Accuracy = {acc:.4f}")

# Matriz de confusión
cm = confusion_matrix(y, y_pred)
# Matriz de confusión
cm = confusion_matrix(y, y_pred)

plt.figure(figsize=(8,8))
ConfusionMatrixDisplay(cm).plot()
plt.show()

## La maldición de la dimensionalidad

**Extra:** Según la teoría, si aumentamos la cantidad de variables explicativas (y por lo tanto la dimensión del espacio de "features"), los puntos tienden a estar a distancias similares, es decir, disminuye la varianza de las distancias entre puntos.

Hagamos una simulación para verificarlo...

In [ ]:
# Este código me lo tiró ChatGPT... estará bien?

import numpy as np
from scipy.spatial.distance import pdist

def distancia_varianza(n, num_puntos=100):
    # Generar puntos aleatorios en [0,1]^n
    puntos = np.random.rand(num_puntos, n)
    
    # Calcular todas las distancias euclidianas por pares (sin repetir)
    distancias = pdist(puntos, metric='euclidean')

    # Calcular y devolver la varianza de las distancias
    return np.var(distancias)

# Ejemplo: calcular para dimensiones 1 a 20
for n in range(1, 20):
    var = distancia_varianza(n)
    print(f"Dimensión {n:2d} -> Varianza de distancias: {var:.6f}")

In [ ]:
# Qué pasó...? No disminuye la varianza??

In [ ]:
n = 100
num_puntos = 100
puntos = np.random.rand(num_puntos, n)
    
# Calcular todas las distancias euclidianas por pares (sin repetir)
distancias = pdist(puntos, metric='euclidean')
distancias

In [ ]:
# Para ver todas las coordenadas agregamos esta opción
np.set_printoptions(threshold=np.inf)  
distancias

Si dividimos el desvío por la media de las distancias, vemos que el desvío tiende a 0.

In [ ]:
def dispersion_relativa(n, num_puntos=100):
    puntos = np.random.rand(num_puntos, n)

    distancias = pdist(puntos, metric='euclidean')

    return np.std(distancias) / np.mean(distancias)

dimensiones = range(1, 101)
valores = [dispersion_relativa(n) for n in dimensiones]

plt.figure(figsize=(8,5))
plt.plot(dimensiones, valores)
plt.xlabel("Dimensión")
plt.ylabel("std(distancias) / mean(distancias)")
plt.title("Concentración de distancias al aumentar la dimensión")
plt.grid(True)
plt.show()